### Testing GraphBuilder

#### Node classification - One Graph - Cora

In [4]:
from torch_geometric import datasets
from rewiring.model.GCN import GCN
from rewiring.pipeline.SingleGraphNodeClassificationPipeline import SingleGraphNodeClassificationPipeline

from rewiring.transform.AddRandomEdges import AddRandomEdges


import torch
import numpy as np
import logging

# logging.basicConfig(level=logging.INFO)

transform = AddRandomEdges(min_density=0.2)
# transform = None
dset = datasets.Planetoid(root=f'./../res/datasets/citeseer', name='CiteSeer', transform=transform)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GCN(num_classes=dset.num_classes, 
            num_features=dset.num_features, 
            hidden_size=16)

model.set_classification_task(task='node')
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
criterion = torch.nn.NLLLoss()

pipeline = SingleGraphNodeClassificationPipeline(
    device=device,
    model=model,
    criterion=criterion,
    optimizer=optimizer
)

pipeline.set_masks(data=dset)

_, _, acc = pipeline.train_validate(
    data=dset[0],
    epochs=200,
    log=True
)
np.array(acc).max()

DEBUG:rewiring.transform.AddRandomEdges:Adding edge...


DEBUG:rewiring.transform.AddRandomEdges:Adding edge...
DEBUG:rewiring.TrainValTestSplitter:Proceeding with single graph splitting method
DEBUG:rewiring.TrainValTestSplitter:Train mask: [2696 2053 2812 ...  403  105 1462]
DEBUG:rewiring.TrainValTestSplitter:Val mask: [2378 3048 2075  425  177 2444 1373 1622 1236 2415 2094 2673  816  990
 2374 1742 2422  150 2442 2229 3051  294 2779  466 2441 3256 2894 3046
  245 2865  755 1342 1892  882 1002  319 1873  868   12 1238  304 1778
 1520   41 2186 2811  616 1150 2397  423 1437 1375 1845  529 2830 2720
 1795 2193 1797 1333 1515 2238 3282  763 2511 3075 1883 2427 2251 3149
 2369 2583 2967  120 2734 1637 1159 3037 2999   91 2196 1871 2833 2149
  590 3025  524 1843 2748  357 1132 2310 2597 2457  369 1127 2828 2660
 1412 1228 1344  509  508 1633 2635 1887  273  711  102 1943 2925 1015
  877 1832 2423 2383 1863 1110 2443 2180 3164  843 2334 2686 2995 2562
 2179 2245  967 2710 2948 1147  400 2343 1424 2022 1277 1408 3066  959
 2539  617 1541 1511 14

Epoch: [1/200]         Training loss: 1.7876         Validation loss: 1.7728         Accuracy: 0.1742
Epoch: [2/200]         Training loss: 1.7791         Validation loss: 1.7634         Accuracy: 0.1742
Epoch: [3/200]         Training loss: 1.7731         Validation loss: 1.7559         Accuracy: 0.1742
Epoch: [4/200]         Training loss: 1.7690         Validation loss: 1.7487         Accuracy: 0.1742
Epoch: [5/200]         Training loss: 1.7650         Validation loss: 1.7434         Accuracy: 0.1772
Epoch: [6/200]         Training loss: 1.7619         Validation loss: 1.7408         Accuracy: 0.2342
Epoch: [7/200]         Training loss: 1.7599         Validation loss: 1.7395         Accuracy: 0.2012
Epoch: [8/200]         Training loss: 1.7586         Validation loss: 1.7395         Accuracy: 0.2012
Epoch: [9/200]         Training loss: 1.7583         Validation loss: 1.7393         Accuracy: 0.2012
Epoch: [10/200]         Training loss: 1.7583         Validation loss: 1.7389     

KeyboardInterrupt: 

#### Node Classification - Multiple Graphs - LRGB PascalVOC-SP

In [3]:
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch_geometric.nn.conv import GCNConv
from torch_geometric.loader import DataLoader
from torch_geometric.datasets import LRGBDataset

from rewiring.model.SimpleGCN import SimpleGCN
from rewiring.transform.AddRandomEdges import AddRandomEdges
from rewiring.transform.ReduceDiameter import ReduceDiameter
from rewiring.pipeline.NodeClassificationPipeline import NodeClassifiactionPipeline

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

batch_size = 64
# transform = ReduceDiameter(max_diam=200)
transform = AddRandomEdges(min_density=0.25)
# transform = None

# Load the dataset
print("Loading and transforming training dataset...")
train_dataset = LRGBDataset(root=f'./../res/datasets/pascal', name='PascalVOC-SP', transform=transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

val_dataset = LRGBDataset(root=f'./../res/datasets/pascal', name='PascalVOC-SP', split='val')
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

test_dataset = LRGBDataset(root=f'./../res/datasets/pascal', name='PascalVOC-SP', split='test')
test_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Create the model
num_features = train_dataset.num_features
num_classes = train_dataset.num_classes
hidden_size = 64
model = SimpleGCN(num_classes=num_classes,
            num_features=num_features,
            hidden_size=hidden_size).to(device)
# model.set_classification_task('graph')

# Set the loss function and optimizer
criterion = nn.NLLLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

pipeline = NodeClassifiactionPipeline(
    device = device,
    model=model,
    criterion=criterion,
    optimizer=optimizer
)

num_epochs = 10
print("Started training...")
pipeline.train_validate(
    train_data=train_loader, 
    val_data=val_loader, 
    epochs=num_epochs, 
    log=True
)
print("Training finished!")

print("Starting testing...")
model.eval()
correct = 0
epoch_test_loss = []
sum = len(test_loader.dataset.y)
print(f'Test element count: {sum}')
for batch in test_loader:
    batch = batch.to(device)
    with torch.no_grad():
        out = model(batch.x, batch.edge_index)
        test_loss = criterion(out, batch.y)
        epoch_test_loss.append(test_loss.item())
        _, pred = torch.max(out.data, 1)
        correct += (pred == batch.y).float().sum().item()

acc = correct / sum
test_loss = np.array(epoch_test_loss).mean().item()

print(f"""Loss: {test_loss:.4f} \
        Accuracy: {acc:.4f}""")

Loading and transforming training dataset...


KeyboardInterrupt: 

#### Graph Classification - Proteins, Mutag

In [ ]:
from torch_geometric import datasets
from torch_geometric.loader import DataLoader
from torch_geometric.transforms import TwoHop
from rewiring.model.GCN import GCN
from rewiring.TrainValTestSplitter import TrainValTestSplitter
from rewiring.transform.ReduceDiameter import ReduceDiameter
from rewiring.transform.AddRandomEdges import AddRandomEdges
from rewiring.pipeline.GraphClassificationPipeline import GraphClassificationPipeline

import torch
import numpy as np
import logging

logging.basicConfig(level=logging.INFO)

transform = None
# transform = AddRandomEdges(min_density=0.4)
# transform = ReduceDiameter(max_diam=26)
# transform = TwoHop()

batch = datasets.TUDataset(root=f'./../res/datasets/proteins', name='PROTEINS', transform=transform, use_node_attr=True, use_edge_attr=True)
# batch = datasets.TUDataset(root=f'./../res/datasets/mutag', name='MUTAG', transform=transform, use_node_attr=True, use_edge_attr=True)
# batch = datasets.TUDataset(root=f'./../res/datasets/enzymes', name='ENZYMES', transform=transform, use_node_attr=True, use_edge_attr=True)
# batch = datasets.TUDataset(root=f'./../res/datasets/collab', name='COLLAB', transform=transform, use_node_attr=True, use_edge_attr=True)
print(f"Number of graphs: {len(batch)}")
print(batch.name)
val_ratio = 0.2
train_dataset, val_dataset, test_dataset = TrainValTestSplitter.graph_class_split(batch, val_size=val_ratio, test_size=1)
val_count = len(val_dataset)

batch_size = 64

train_dloader: DataLoader = DataLoader(train_dataset, 
                                       shuffle=True, 
                                       batch_size=batch_size)

val_dloader: DataLoader = DataLoader(val_dataset, 
                                      shuffle=False, 
                                      batch_size=batch_size)

print("Creating model...")
model = GCN(num_classes=batch.num_classes, 
            num_features=batch.num_features, 
            hidden_size=16)
model.set_classification_task(task='graph')
print("Model created!")

optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
criterion = torch.nn.NLLLoss()

pipeline = GraphClassificationPipeline(
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    model=model,
    criterion=criterion,
    optimizer=optimizer
)

_, _, acc = pipeline.train_validate(
    train_data=train_dloader,
    val_data=val_dloader,
    epochs=500,
    log=True
)
np.array(acc).max()

In [ ]:
import logging
from torch_geometric import datasets
from rewiring.TrainValTestSplitter import TrainValTestSplitter

logging.basicConfig(level=logging.DEBUG)

dset = datasets.Planetoid(root=f'./../res/datasets/citeseer', name='CiteSeer')
TrainValTestSplitter.single_graph_split(dset=dset, val_size=0.2, test_size=1)

In [2]:
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch_geometric.nn.conv import GCNConv
from torch_geometric.loader import DataLoader
from torch_geometric.datasets import LRGBDataset

from rewiring.model.SimpleGCN import SimpleGCN
from rewiring.transform.AddRandomEdges import AddRandomEdges
from rewiring.transform.ReduceDiameter import ReduceDiameter
from rewiring.pipeline.NodeClassificationPipeline import NodeClassifiactionPipeline
from rewiring.TrainValTestSplitter import TrainValTestSplitter
from graphgen.SparseGraphDataset import SparseGraphDataset

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

batch_size = 64
# transform = ReduceDiameter(max_diam=200)
# transform = AddRandomEdges(min_density=0.25)
transform = None

# Load the dataset
print("Loading and transforming training dataset...")
dset = SparseGraphDataset(root=f'./../res/datasets/sparse_dset', transform=transform)

train_dataset, val_dataset, _ = TrainValTestSplitter.graph_class_split(dset)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader =  DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Create the model
num_features = dset.num_features
num_classes = 5
hidden_size = 64
model = SimpleGCN(num_classes=num_classes,
            num_features=num_features,
            hidden_size=hidden_size).to(device)
# model.set_classification_task('graph')

# Set the loss function and optimizer
criterion = nn.NLLLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

pipeline = NodeClassifiactionPipeline(
    device = device,
    model=model,
    criterion=criterion,
    optimizer=optimizer
)

num_epochs = 10
print("Started training...")
pipeline.train_validate(
    train_data=train_loader, 
    val_data=val_loader, 
    epochs=num_epochs, 
    log=True
)
print("Training finished!")

Loading and transforming training dataset...
This function shouldn't be called, the datasets should 
            have been downloaded to their correct folder during the cloning process!e
            Repository link: https://git01lab.cs.univie.ac.at/peteri00/rewiring-heuristics.git
Started training...


Processing...
Done!


AttributeError: 'list' object has no attribute 'size'